## PASO 2: Iterar lista de URLs con un Web Scraper considerando URLs(pages)

### Paginar cada URL (cada URL tiene un parámetro de paginación) y extraer los datos de cada página.
Necesitamos saber cuantas paginas tiene cada URL. Guardar cada version del URL con el número de página correspondiente para luego scrapear cada una de esas páginas.

#### Ya que tenemos la URL de cada página, iterar sobre cada una de ellas y extraer los datos necesarios para construir otro set de URLs los cuales contienen otra etapa de informacion de identificacion individual de cada registro. En este punto, buscamos pasar de numero de expediente(tipo de aplicacion, año) al numero de registro (ID unico) de cada marca.

In [ ]:
import json
import requests

#Los URLs se contienen en un diccionario de diccionarios incia;
with open("1.rangos_busquedas.json", encoding="utf-8") as f:
    rangos_urls = json.load(f)

tuplas_urls = [
    (llamada, datos["url"], datos.get("count", 0))
    for llamada, datos in rangos_urls.items()
]

# Agregar valor de conteo/100 para saber el numero de paginas a iterar por cada URL
tuplas_urls = [(llamada, url, count, 0, count//100) for llamada, url, count in tuplas_urls]
urls_paginas = []

for llamada, url_base, count, _, max_pagina in tuplas_urls:
    for p in range(max_pagina + 1):
        urls_paginas.append(f"{url_base}&page={p}")



# reiniciar la funcion de crear sesion
def crear_sesion():
    session = requests.Session()
    session.get("https://marcia.impi.gob.mx/marcas/search/quick")
    xsrf_token = session.cookies.get("XSRF-TOKEN")
    headers = {
        "X-XSRF-TOKEN": xsrf_token,
        "Content-Type": "application/json",
        "Referer": "https://marcia.impi.gob.mx/marcas/search/quick",
    }
    return session, headers

# funcion extracion numero 2 (urls paginados)
def extraer_result_page(url, session, headers):
    search_id   = url.split("?s=")[1].split("&")[0]
    page_number = int(url.split("&page=")[1])
    
    payload = {
        "searchId":         search_id,
        "pageSize":         100, # el máximo permitido por la API
        "pageNumber":       page_number,
        "sort":             "",
        "statusFilter":     [],
        "viennaCodeFilter": [],
        "niceClassFilter":  []
    }
    
    respuesta = session.post(
        "https://marcia.impi.gob.mx/marcas/search/internal/result",
        json=payload,
        headers=headers
    )
    return respuesta.json()["resultPage"]


todos_los_registros = []
session, headers = crear_sesion() 

for url in urls_paginas:
    print(f"Procesando: {url}")
    resultado = extraer_result_page(url, session, headers)
    todos_los_registros.extend(resultado)
    print(f"  Registros: {len(resultado)} | Acumulado: {len(todos_los_registros)}")


# guardar al terminar el loop
nombre_archivo = "2.MARCIA_result_pages.json"

with open(nombre_archivo, "w", encoding="utf-8") as f:
    json.dump(todos_los_registros, f, ensure_ascii=False, indent=2)

print(f"\nGuardado: {nombre_archivo} con {len(todos_los_registros)} registros totales")



Procesando: https://marcia.impi.gob.mx/marcas/search/result?s=99cb7bd8-b88d-492f-9d11-86a38bcc6150&m=l&page=0
  Registros: 100 | Acumulado: 100
Procesando: https://marcia.impi.gob.mx/marcas/search/result?s=99cb7bd8-b88d-492f-9d11-86a38bcc6150&m=l&page=1
  Registros: 100 | Acumulado: 200
Procesando: https://marcia.impi.gob.mx/marcas/search/result?s=99cb7bd8-b88d-492f-9d11-86a38bcc6150&m=l&page=2
  Registros: 100 | Acumulado: 300
Procesando: https://marcia.impi.gob.mx/marcas/search/result?s=99cb7bd8-b88d-492f-9d11-86a38bcc6150&m=l&page=3
  Registros: 100 | Acumulado: 400
Procesando: https://marcia.impi.gob.mx/marcas/search/result?s=99cb7bd8-b88d-492f-9d11-86a38bcc6150&m=l&page=4
  Registros: 100 | Acumulado: 500
Procesando: https://marcia.impi.gob.mx/marcas/search/result?s=99cb7bd8-b88d-492f-9d11-86a38bcc6150&m=l&page=5
  Registros: 100 | Acumulado: 600
Procesando: https://marcia.impi.gob.mx/marcas/search/result?s=99cb7bd8-b88d-492f-9d11-86a38bcc6150&m=l&page=6
  Registros: 100 | Acumula

In [3]:
### JSON Marcia Results Pages (DF V.1)

import json
import pandas as pd

with open("2.MARCIA_result_pages.json", encoding="utf-8") as f:
    datos = json.load(f)

# aplana todo automáticamente — sin hardcodear nada
df = pd.json_normalize(datos, sep=".")

# owners es lista — aplanar a string
df["owners"] = df["owners"].apply(
    lambda x: "; ".join(x) if isinstance(x, list) else x
)

# classes es lista — aplanar a string
df["classes"] = df["classes"].apply(
    lambda x: "; ".join(str(c) for c in x) if isinstance(x, list) else x
)

df = df.sort_values("registrationNumber")
print(df.shape)
print(df.columns.tolist())
#print(df.head())


#objetos lista identificadores unicos
registration_numbers = df["registrationNumber"].dropna().unique().tolist()
application_numbers = df["applicationNumber"].dropna().unique().tolist()
print(len(registration_numbers))
print(len(application_numbers))

(153978, 15)
['id', 'applicationNumber', 'registrationNumber', 'title', 'status', 'appType', 'owners', 'goodsAndServices', 'classes', 'images', 'dates.application', 'dates.publication', 'dates.cancellation', 'dates.expiry', 'dates.registration']
153977
153978


In [5]:
df.to_csv("2.MARCIA_result_pages.csv", index=False, encoding="utf-8-sig")